# Resume Data Cleaning

Clean and standardize resume data with predefined IT skills categorization

In [6]:
import json
import pandas as pd
import numpy as np
from collections import Counter

print("Libraries loaded successfully")

Libraries loaded successfully


In [7]:
# Load the parsed resume data
with open('data/processed/parsed_resumes.json', 'r') as f:
    data = json.load(f)[:100]

print(f"Loaded {len(data)} resumes")

# Count successfully parsed resumes
parsed_count = sum(1 for resume in data if resume.get('parsed') is not None)
print(f"Successfully parsed: {parsed_count} ({parsed_count/len(data)*100:.1f}%)")

Loaded 100 resumes
Successfully parsed: 100 (100.0%)


In [8]:
from clean_skills import clean_skills, find_best_language_match, find_best_skill_match, find_best_soft_skill_match, load_language_skills, load_technology_skills, load_soft_skills
from clean_designations import clean_designations, find_best_designation_match, load_occupation_titles

print("Cleaning skills and designations for all resumes...")

original_skills_count = 0
cleaned_it_skills_count = 0
cleaned_soft_skills_count = 0
cleaned_languages_count = 0

original_designations_count = 0
cleaned_designations_count = 0

unmatched_skills = Counter()
unmatched_designations = Counter()

all_titles, title_lookup = load_occupation_titles()

technology_skills, technology_skill_to_category = load_technology_skills()
soft_skills, soft_skill_to_category = load_soft_skills()
language_skills, language_skill_to_category = load_language_skills()

for i, resume in enumerate(data):

    if resume.get('parsed') and 'skills' in resume['parsed']:
        original_skills = resume['parsed']['skills']

        if isinstance(original_skills, list):
            original_skills_count += len(original_skills)

            it_skills, it_categories, soft_skills, soft_categories, languages, lang_categories = clean_skills(original_skills)

            cleaned_it_skills_count += len(it_skills)
            cleaned_soft_skills_count += len(soft_skills)
            cleaned_languages_count += len(languages)

            for skill in original_skills:
                if isinstance(skill, str) and len(skill.strip()) >= 2:
                    it_match = find_best_skill_match(skill, technology_skills, technology_skill_to_category)
                    soft_match = find_best_soft_skill_match(skill, soft_skills, soft_skill_to_category)
                    lang_match = find_best_language_match(skill, language_skills, language_skill_to_category)
                    if not it_match and not soft_match and not lang_match:
                        unmatched_skills[skill.lower().strip()] += 1

            resume['parsed']['skills_original'] = original_skills
            resume['parsed']['it_skills'] = it_skills
            resume['parsed']['it_skill_categories'] = it_categories
            resume['parsed']['soft_skills'] = soft_skills
            resume['parsed']['soft_skill_categories'] = soft_categories
            resume['parsed']['languages'] = languages
            resume['parsed']['language_categories'] = lang_categories
            resume['parsed']['skills'] = it_skills

    if resume.get('parsed') and 'designition' in resume['parsed']:
        original_designations = resume['parsed']['designition']

        if isinstance(original_designations, list):
            original_designations_count += len(original_designations)

            titles, codes, categories = clean_designations(original_designations)
            cleaned_designations_count += len(titles)

            for title in original_designations:
                if isinstance(title, str) and len(title.strip()) >= 3:
                    match = find_best_designation_match(title, all_titles, title_lookup)
                    if not match:
                        unmatched_designations[title.lower().strip()] += 1

            resume['parsed']['designations_original'] = original_designations
            resume['parsed']['designations'] = titles
            resume['parsed']['designation_codes'] = codes
            resume['parsed']['designation_categories'] = categories

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(data)} resumes")

print(f"\nCleaning completed!")

print(f"\nSkills:")
print(f"Original skills: {original_skills_count}")
print(f"Cleaned IT skills: {cleaned_it_skills_count}")
print(f"Cleaned soft skills: {cleaned_soft_skills_count}")
print(f"Cleaned language skills: {cleaned_languages_count}")
print(f"Total cleaned skills: {cleaned_it_skills_count + cleaned_soft_skills_count + cleaned_languages_count}")
print(f"IT skill retention rate: {cleaned_it_skills_count/original_skills_count*100:.1f}%")
print(f"Soft skill retention rate: {cleaned_soft_skills_count/original_skills_count*100:.1f}%")
print(f"Language skill retention rate: {cleaned_languages_count/original_skills_count*100:.1f}%")
print(f"Total retention rate: {(cleaned_it_skills_count + cleaned_soft_skills_count + cleaned_languages_count)/original_skills_count*100:.1f}%")

print(f"\nDesignations:")
print(f"Original designations: {original_designations_count}")
print(f"Cleaned designations: {cleaned_designations_count}")
print(f"Designation retention rate: {cleaned_designations_count/original_designations_count*100:.1f}%" if original_designations_count else "Designation retention rate: 0%")

print(f"\nTop 15 unmatched skills:")
for skill, count in unmatched_skills.most_common(15):
    print(f"  {skill}: {count}")

print(f"\nTop 15 unmatched designations:")
for title, count in unmatched_designations.most_common(15):
    print(f"  {title}: {count}")


Cleaning skills and designations for all resumes...


/Users/hammadhassan/Documents/semester4/career-mentor-ai/resume_processor/scripts/extract_occupations.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anzsco_df = anzsco_df[anzsco_df["Occupation Code"].str.isdigit().fillna(False)]
/Users/hammadhassan/Documents/semester4/career-mentor-ai/resume_processor/scripts/extract_occupations.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anzsco_df = anzsco_df[anzsco_df["Occupation Code"].str.isdigit().fillna(False)]
/Users/hammadhassan/Documents/semester4/career-mentor-ai/resume_processor/scripts/extract_

Processed 100/100 resumes

Cleaning completed!

Skills:
Original skills: 2840
Cleaned IT skills: 1092
Cleaned soft skills: 118
Cleaned language skills: 140
Total cleaned skills: 1350
IT skill retention rate: 38.5%
Soft skill retention rate: 4.2%
Language skill retention rate: 4.9%
Total retention rate: 47.5%

Designations:
Original designations: 348
Cleaned designations: 47
Designation retention rate: 13.5%

Top 15 unmatched skills:
  birth: 23
  hobbies: 12
  it: 10
  religion: 9
  cricket: 6
  avenue: 5
  sh: 4
  participation: 4
  motors: 4
  india: 4
  switches: 3
  credentials: 3
  surfing: 3
  electricity: 3
  nationality :indian: 3

Top 15 unmatched designations:
  supervisor: 12
  representative: 9
  student: 9
  autocad: 9
  team leader: 9
  associate: 8
  team member: 7
  director: 6
  quality control: 6
  civil engineer: 5
  project engineer: 5
  accountant: 4
  data entry operator: 3
  entry operator: 3
  partner: 3


/Users/hammadhassan/Documents/semester4/career-mentor-ai/resume_processor/scripts/extract_occupations.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anzsco_df = anzsco_df[anzsco_df["Occupation Code"].str.isdigit().fillna(False)]


In [9]:
# Generate cleaning report
def generate_cleaning_report():
    """Generate a comprehensive cleaning report including IT, soft, language skills and designations"""
    
    # IT Skills analysis
    all_cleaned_it_skills = []
    all_it_categories = []
    it_skills_per_resume = []
    
    # Soft Skills analysis
    all_cleaned_soft_skills = []
    all_soft_categories = []
    soft_skills_per_resume = []
    
    # Language Skills analysis
    all_cleaned_languages = []
    all_language_categories = []
    languages_per_resume = []

    # Designations analysis
    all_cleaned_designations = []
    all_designation_categories = []
    designations_per_resume = []
    
    # Other metrics
    experience_values = []
    companies_count = []
    universities_count = []
    
    for resume in data:
        if resume.get('parsed'):
            parsed = resume['parsed']
            
            # IT Skills
            if 'it_skills' in parsed and isinstance(parsed['it_skills'], list):
                it_skills = parsed['it_skills']
                all_cleaned_it_skills.extend(it_skills)
                it_skills_per_resume.append(len(it_skills))
                
                if 'it_skill_categories' in parsed:
                    all_it_categories.extend(parsed['it_skill_categories'])
            elif 'skills' in parsed and isinstance(parsed['skills'], list):
                skills = parsed['skills']
                all_cleaned_it_skills.extend(skills)
                it_skills_per_resume.append(len(skills))
                
                if 'skill_categories' in parsed:
                    all_it_categories.extend(parsed['skill_categories'])
            else:
                it_skills_per_resume.append(0)
            
            # Soft Skills
            if 'soft_skills' in parsed and isinstance(parsed['soft_skills'], list):
                soft_skills = parsed['soft_skills']
                all_cleaned_soft_skills.extend(soft_skills)
                soft_skills_per_resume.append(len(soft_skills))
                
                if 'soft_skill_categories' in parsed:
                    all_soft_categories.extend(parsed['soft_skill_categories'])
            else:
                soft_skills_per_resume.append(0)
            
            # Language Skills
            if 'languages' in parsed and isinstance(parsed['languages'], list):
                languages = parsed['languages']
                all_cleaned_languages.extend(languages)
                languages_per_resume.append(len(languages))
                
                if 'language_categories' in parsed:
                    all_language_categories.extend(parsed['language_categories'])
            else:
                languages_per_resume.append(0)

            # Designations
            if 'designations' in parsed and isinstance(parsed['designations'], list):
                designations = parsed['designations']
                all_cleaned_designations.extend(designations)
                designations_per_resume.append(len(designations))

                if 'designation_categories' in parsed:
                    all_designation_categories.extend(parsed['designation_categories'])
            else:
                designations_per_resume.append(0)
            
            # Experience
            if 'total_exp' in parsed:
                experience_values.append(parsed['total_exp'])
            
            # Companies
            if 'Companies worked at' in parsed and isinstance(parsed['Companies worked at'], list):
                companies_count.append(len(parsed['Companies worked at']))
            
            # Universities
            if 'university' in parsed and isinstance(parsed['university'], list):
                universities_count.append(len(parsed['university']))
    
    total_skills_per_resume = [
        it + soft + lang for it, soft, lang in 
        zip(it_skills_per_resume, soft_skills_per_resume, languages_per_resume)
    ]
    
    report = {
        "total_resumes": len(data),
        "parsed_resumes": sum(1 for r in data if r.get('parsed')),
        "it_skills_stats": {
            "total_unique_skills": len(set(all_cleaned_it_skills)),
            "total_skill_mentions": len(all_cleaned_it_skills),
            "avg_skills_per_resume": np.mean(it_skills_per_resume) if it_skills_per_resume else 0,
            "top_skills": dict(Counter(all_cleaned_it_skills).most_common(10)),
            "skills_by_category": dict(Counter(all_it_categories))
        },
        "soft_skills_stats": {
            "total_unique_skills": len(set(all_cleaned_soft_skills)),
            "total_skill_mentions": len(all_cleaned_soft_skills),
            "avg_skills_per_resume": np.mean(soft_skills_per_resume) if soft_skills_per_resume else 0,
            "top_skills": dict(Counter(all_cleaned_soft_skills).most_common(10)),
            "skills_by_category": dict(Counter(all_soft_categories))
        },
        "language_skills_stats": {
            "total_unique_languages": len(set(all_cleaned_languages)),
            "total_language_mentions": len(all_cleaned_languages),
            "avg_languages_per_resume": np.mean(languages_per_resume) if languages_per_resume else 0,
            "top_languages": dict(Counter(all_cleaned_languages).most_common(10)),
            "languages_by_category": dict(Counter(all_language_categories))
        },
        "designation_stats": {
            "total_unique_designations": len(set(all_cleaned_designations)),
            "total_designation_mentions": len(all_cleaned_designations),
            "avg_designations_per_resume": np.mean(designations_per_resume) if designations_per_resume else 0,
            "top_designations": dict(Counter(all_cleaned_designations).most_common(10)),
            "designations_by_category": dict(Counter(all_designation_categories))
        },
        "combined_skills_stats": {
            "total_unique_skills": len(set(
                all_cleaned_it_skills +
                all_cleaned_soft_skills +
                all_cleaned_languages
            )),
            "total_skill_mentions": (
                len(all_cleaned_it_skills) +
                len(all_cleaned_soft_skills) +
                len(all_cleaned_languages)
            ),
            "avg_total_skills_per_resume": np.mean(total_skills_per_resume) if total_skills_per_resume else 0
        },
        "experience_stats": {
            "avg_experience": np.mean(experience_values) if experience_values else 0,
            "median_experience": np.median(experience_values) if experience_values else 0,
            "max_experience": max(experience_values) if experience_values else 0,
            "resumes_with_experience": sum(1 for exp in experience_values if exp > 0)
        },
        "companies_stats": {
            "avg_companies_per_resume": np.mean(companies_count) if companies_count else 0,
            "resumes_with_companies": sum(1 for c in companies_count if c > 0)
        },
        "education_stats": {
            "avg_universities_per_resume": np.mean(universities_count) if universities_count else 0,
            "resumes_with_university": sum(1 for u in universities_count if u > 0)
        }
    }
    
    return report


# Generate and display report
report = generate_cleaning_report()

print("Data Cleaning Report with IT, Soft, Language Skills and Designations")
print("=" * 60)
print(f"Total Resumes: {report['total_resumes']}")
print(f"Parsed Resumes: {report['parsed_resumes']}")

print(f"\n{'='*60}")
print("DESIGNATION STATISTICS")
print(f"{'='*60}")
print(f"  Total Unique Designations: {report['designation_stats']['total_unique_designations']}")
print(f"  Total Designation Mentions: {report['designation_stats']['total_designation_mentions']}")
print(f"  Avg Designations per Resume: {report['designation_stats']['avg_designations_per_resume']:.2f}")

print(f"\nTop 10 Designations:")
for title, count in report['designation_stats']['top_designations'].items():
    print(f"  {title}: {count}")

print(f"\nDesignations by Category:")
for category, count in sorted(
    report['designation_stats']['designations_by_category'].items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"  {category}: {count}")


Data Cleaning Report with IT, Soft, Language Skills and Designations
Total Resumes: 100
Parsed Resumes: 100

DESIGNATION STATISTICS
  Total Unique Designations: 16
  Total Designation Mentions: 47
  Avg Designations per Resume: 0.47

Top 10 Designations:
  Computer Systems Technician: 9
  Software Engineer: 8
  Operator Command Support Systems (Army): 6
  ICT Quality Assurance Engineer: 6
  Quality Analyst (ICT): 3
  ICT Security Project Manager: 2
  Database Analyst: 2
  Chief Information Security Officer: 2
  Devops Engineer: 2
  IT Service Delivery Manager: 1

Designations by Category:
  Principal Title: 19
  Specialisation: 13
  Occupation in nec category: 10
  Alternative Title: 5


## Retaining only the essential data for model training

In [ ]:
import json

# Input and output file paths
input_file = "./data/processed/cleaned_resumes.json"
output_file = "./data/processed/essential_resumes.json"

# Load the JSON data
with open(input_file, "r", encoding="utf-8") as f:
    resumes = json.load(f)

essential_resumes = []

for resume in resumes:
    parsed = resume.get("parsed", {})

    # Remove duplicates from it_skill_categories while preserving order
    it_skill_categories = parsed.get("it_skill_categories", [])
    if isinstance(it_skill_categories, list):
        seen = set()
        it_skill_categories = [
            x for x in it_skill_categories
            if not (x in seen or seen.add(x))
        ]

    essential_data = {
        "skills": parsed.get("skills"),
        "it_skills": parsed.get("it_skills"),
        "it_skill_categories": it_skill_categories,
        "soft_skills": parsed.get("soft_skills"),
        "languages": parsed.get("languages"),
        "designations": parsed.get("designations"),
    }

    essential_resumes.append(essential_data)

# Save the cleaned data to a new JSON file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(essential_resumes, f, indent=4, ensure_ascii=False)

print(f"Saved {len(essential_resumes)} cleaned resumes to '{output_file}'")


FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_resumes.json'